# Multi-Layer Perceptron From Scratch
## Multi-Class MIDI Note Classification on GuitarSet Audio Frames

**Course:** CMOR 438 — Machine Learning  
**Dataset:** GuitarSet (`audio_hex_cln`, all 6 strings, 10 tracks)  
**Task:** Given 18 audio features extracted from a 46 ms guitar frame, identify *which* MIDI note is being played (12-class classification, top-12 most frequent notes).

## 1. Introduction

### 1.1 From Perceptron to Deep Networks

The Perceptron solves binary, linearly-separable tasks. Music transcription — deciding *which* note is playing — requires distinguishing 12 or more classes that share overlapping spectral features. No single hyperplane can separate them reliably.

The **Multi-Layer Perceptron (MLP)** stacks multiple *layers* of neurons, each applying a nonlinear activation after a linear projection. The stacked nonlinearities can represent arbitrarily complex decision surfaces (Universal Approximation Theorem, Cybenko 1989).

### 1.2 Architecture

An MLP with one hidden layer of width $H$ computes:

$$\mathbf{h} = \sigma\!\left(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1\right) \in \mathbb{R}^H$$
$$\hat{\mathbf{y}} = f_{\text{out}}\!\left(\mathbf{W}_2 \mathbf{h} + \mathbf{b}_2\right) \in \mathbb{R}^C$$

where $\sigma$ is the hidden activation (ReLU), $f_{\text{out}}$ is the output activation, $C$ is the number of classes, and $\hat{\mathbf{y}}_c$ is the predicted score for class $c$.

For multi-class output, $\text{softmax}(\mathbf{z})_c = e^{z_c} / \sum_k e^{z_k}$ converts raw scores to a proper probability distribution. In this implementation we use MSE with raw outputs (Softmax backprop is not yet implemented) and apply softmax post-hoc for visualisation.

### 1.3 Why this task is harder than binary detection

- Notes on different strings can have the same MIDI pitch (guitar re-entrant tuning means different strings share harmonic structure)
- Notes in the same octave have harmonic overlap in MFCCs
- A 12-class F1 requires the model to distinguish neighbours in pitch space, not just signal vs. noise

## 2. Algorithm

### 2.1 Forward Pass

For a network with $L$ layers, the forward pass propagates the input $\mathbf{x}$ through each layer:

$$\mathbf{a}^{(0)} = \mathbf{x}$$
$$\mathbf{z}^{(l)} = \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}, \quad l = 1, \ldots, L$$
$$\mathbf{a}^{(l)} = \sigma^{(l)}\!\left(\mathbf{z}^{(l)}\right)$$

The prediction is $\hat{\mathbf{y}} = \mathbf{a}^{(L)}$.

### 2.2 Loss Function

For one-hot encoded targets $\mathbf{y} \in \{0,1\}^C$ and predictions $\hat{\mathbf{y}} \in \mathbb{R}^C$, the mean squared error (MSE) over a batch of $N$ samples is:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^N \|\mathbf{y}_i - \hat{\mathbf{y}}_i\|^2$$

### 2.3 Backpropagation

Backpropagation applies the **chain rule** in reverse order through the network. Starting from the output layer:

$$\boldsymbol{\delta}^{(L)} = \nabla_{\hat{\mathbf{y}}} \mathcal{L} \odot \sigma^{(L)}{}'\!\left(\mathbf{z}^{(L)}\right)$$

For hidden layers $l = L-1, \ldots, 1$:

$$\boldsymbol{\delta}^{(l)} = \left(\mathbf{W}^{(l+1)\top} \boldsymbol{\delta}^{(l+1)}\right) \odot \sigma^{(l)}{}'\!\left(\mathbf{z}^{(l)}\right)$$

Parameter gradients:

$$\nabla_{\mathbf{W}^{(l)}} \mathcal{L} = \frac{1}{N} \boldsymbol{\delta}^{(l)} \mathbf{a}^{(l-1)\top}$$
$$\nabla_{\mathbf{b}^{(l)}} \mathcal{L} = \frac{1}{N} \sum_{i=1}^N \boldsymbol{\delta}^{(l)}_i$$

### 2.4 SGD Weight Update

Stochastic Gradient Descent updates each parameter by a small step in the direction of steepest descent:

$$\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$$

where $\eta$ is the learning rate. With mini-batches, the gradient is estimated over a random subset of the training data — this adds noise that can help escape shallow local minima.

## 3. Imports

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent.parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier as SklearnMLP
from sklearn.preprocessing import StandardScaler as SklearnScaler
from sklearn.decomposition import PCA

from rice_Ml.supervised_ml import MLP
from rice_Ml.activations import ReLU, Softmax
from rice_Ml.loss import MeanSquaredError
from rice_Ml.optimizers import SGD
from rice_Ml.preprocessing.dataset import load_dataset
from rice_Ml.preprocessing.scale import StandardScaler
from rice_Ml.model_selection.split import train_test_split
from rice_Ml.metrics import accuracy, precision, recall, f1_score

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Imports OK")

## 4. Load Data

Features were pre-extracted from 10 GuitarSet tracks, all 6 strings, using `scripts/extract_example_features.py`. Silent frames (MIDI label 0) were removed. Only the **top 12 most frequent MIDI notes** were retained to keep the class set manageable; labels were then remapped to contiguous integers 0–11.

In [ ]:
NPZ_PATH = Path("guitarset_features.npz")
X, y = load_dataset(NPZ_PATH)

N_CLASSES = len(np.unique(y))
FEATURE_NAMES = [
    "RMS", "ZCR", "Centroid", "Bandwidth", "Rolloff",
    *[f"MFCC {i}" for i in range(1, 14)],
]

print(f"X shape    : {X.shape}  (n_frames × n_features)")
print(f"y shape    : {y.shape}")
print(f"Classes    : {np.unique(y)}")
print(f"N classes  : {N_CLASSES}")

counts = np.bincount(y)
print("\nClass distribution:")
for c, n in enumerate(counts):
    print(f"  Class {c:2d}: {n:5d} frames ({n/len(y):.1%})")

**Interpretation:** The 12 classes are reasonably balanced — the largest class has ~15% of frames and the smallest ~6%. This is much more balanced than the voiced/silent task, so accuracy is a more meaningful metric here. Nonetheless, per-class F1 will reveal if any notes are systematically confused.

## 5. Exploratory Data Analysis

In [ ]:
# --- 5.1  Class distribution bar chart ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N_CLASSES), counts, color="#4C72B0", alpha=0.85)
ax.set_xlabel("Class label (remapped from MIDI note)")
ax.set_ylabel("Frame count")
ax.set_title("Class Distribution (Top-12 MIDI Notes, All 6 Strings)", fontweight="bold")
ax.set_xticks(range(N_CLASSES))
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.2  Mean feature values per class (MFCC 1–3 shown) ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for col_idx, (ax, feat_idx, feat_name) in enumerate(zip(axes, [0, 2, 5], ["RMS", "Spectral Centroid", "MFCC 1"])):
    class_means = [X[y == c, feat_idx].mean() for c in range(N_CLASSES)]
    class_stds  = [X[y == c, feat_idx].std()  for c in range(N_CLASSES)]
    ax.bar(range(N_CLASSES), class_means, yerr=class_stds,
           capsize=3, color="#4C72B0", alpha=0.8, error_kw={"linewidth": 0.8})
    ax.set_title(feat_name, fontweight="bold")
    ax.set_xlabel("Class")
    if col_idx == 0:
        ax.set_ylabel("Mean ± std (raw)")
    ax.set_xticks(range(N_CLASSES))

plt.suptitle("Selected Feature Means by Class", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** MFCC 1 (the cepstral coefficient capturing the overall spectral envelope shape) varies more across classes than RMS or spectral centroid. Higher-pitched notes (higher class indices) tend to have higher spectral centroids — physically expected, since the fundamental frequency and its harmonics shift upward. RMS is roughly uniform across classes because all these frames are voiced (silent frames were removed).

In [ ]:
# --- 5.3  MFCC heatmap: class-mean MFCCs (features 5–17) ---
mfcc_by_class = np.array([X[y == c, 5:18].mean(axis=0) for c in range(N_CLASSES)])

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mfcc_by_class, aspect="auto", cmap="RdBu_r")
plt.colorbar(im, ax=ax)
ax.set_xlabel("MFCC coefficient index")
ax.set_ylabel("Class")
ax.set_xticks(range(13))
ax.set_xticklabels([f"MFCC {i+1}" for i in range(13)], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(N_CLASSES))
ax.set_title("Mean MFCC Values per Class", fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation:** The MFCC heatmap shows that the 12 classes produce distinct spectral envelope "fingerprints". MFCC 1 (first column) varies most — this is the dominant timbre coefficient and captures the bulk of the pitch-related spectral shape. Higher-order MFCCs capture finer envelope detail; their class-mean differences are smaller but still discriminative. The MLP will learn a nonlinear combination of all 13 MFCCs (plus the 5 energy/spectral features) to separate the classes.

In [ ]:
# --- 5.4  PCA scatter (first 3 components) ---
N_VIZ = 2000
idx_viz = rng.choice(len(X), N_VIZ, replace=False)
pca = PCA(n_components=3, random_state=SEED)
X_pca = pca.fit_transform(X[idx_viz])
y_viz = y[idx_viz]

cmap = plt.cm.get_cmap("tab20", N_CLASSES)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (xi, yi, xlabel, ylabel) in zip(axes, [
    (0, 1, f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
           f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"),
    (0, 2, f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
           f"PC3 ({pca.explained_variance_ratio_[2]:.1%})"),
]):
    sc = ax.scatter(X_pca[:, xi], X_pca[:, yi], c=y_viz,
                    cmap=cmap, vmin=-0.5, vmax=N_CLASSES - 0.5,
                    s=10, alpha=0.5, edgecolors="none")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"PC{xi+1} vs PC{yi+1}", fontweight="bold")

plt.colorbar(sc, ax=axes, label="Class", ticks=range(N_CLASSES))
plt.suptitle("PCA Projection — Colour = Note Class", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"PC1–PC3 explain {sum(pca.explained_variance_ratio_[:3]):.1%} of total variance.")

**Interpretation:** The PCA scatter reveals that the 12 note classes are **not** linearly separable in the principal components — they form overlapping clouds rather than clean clusters. This confirms that a Perceptron (single hyperplane) would struggle at this task, and that the MLP's nonlinear hidden layers are necessary. The classes do show some structure: adjacent class indices (similar MIDI pitches) tend to cluster nearby in PCA space, which reflects the gradual change in spectral content across pitches.

## 6. Preprocessing

Two steps:

1. **Standardise** — feature means to 0, variances to 1. Required for stable gradient flow through deep layers.

2. **One-hot encode** — the MLP expects targets as probability vectors $\mathbf{y} \in \{0,1\}^C$. We convert integer class labels $y \in \{0, \ldots, 11\}$ to one-hot via $\mathbf{y} = \mathbf{I}_C[y]$.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train_sc = scaler.fit(X_train).transform(X_train)
X_test_sc  = scaler.transform(X_test)

# One-hot encode
def to_onehot(labels, n_classes):
    oh = np.zeros((len(labels), n_classes))
    oh[np.arange(len(labels)), labels] = 1.0
    return oh

y_train_oh = to_onehot(y_train, N_CLASSES)
y_test_oh  = to_onehot(y_test,  N_CLASSES)

print(f"Train : {X_train_sc.shape[0]:,} samples")
print(f"Test  : {X_test_sc.shape[0]:,} samples")
print(f"y_train_oh shape: {y_train_oh.shape}")

## 7. Train the MLP

In [ ]:
model = MLP(
    hidden_layers=[64, 32],
    activation=ReLU(),
    output_activation=ReLU(),    # Softmax applied post-hoc for visualisation
    loss=MeanSquaredError(),
    optimizer=SGD(learning_rate=0.01),
    n_epochs=80,
    batch_size=256,
    random_state=SEED,
)

model.fit(X_train_sc, y_train_oh)

print(f"Training complete. Loss history length: {len(model.loss_history_)} epochs.")
print(f"Initial loss : {model.loss_history_[0]:.4f}")
print(f"Final loss   : {model.loss_history_[-1]:.4f}")

In [ ]:
# --- Plot loss curve ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(model.loss_history_) + 1), model.loss_history_,
        linewidth=2, color="#4C72B0")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean Squared Error (training)")
ax.set_title("MLP Training Loss Curve", fontweight="bold")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

**Interpretation:** The loss should decrease monotonically (or nearly so) if the learning rate is appropriate. A log-scale y-axis makes it easier to see early rapid improvement alongside later fine-tuning. If the curve plateaus early, increasing the learning rate or adding more hidden units would help. If it oscillates, the learning rate is too high.

## 8. Evaluate

In [ ]:
# Predictions: raw outputs → argmax for class label
y_raw_train = model.predict(X_train_sc)          # shape (n_train, 12)
y_raw_test  = model.predict(X_test_sc)            # shape (n_test, 12)

y_pred_train = np.argmax(y_raw_train, axis=1)
y_pred_test  = np.argmax(y_raw_test,  axis=1)

train_acc = accuracy(y_train, y_pred_train)
test_acc  = accuracy(y_test,  y_pred_test)

print(f"Training accuracy : {train_acc:.4f}")
print(f"Test accuracy     : {test_acc:.4f}")

# Per-class F1
print("\nPer-class F1 (one-vs-rest):")
print(f"{'Class':<8} {'Precision':>10} {'Recall':>9} {'F1':>8}")
print("-" * 40)
for c in range(N_CLASSES):
    y_true_bin = (y_test == c).astype(int)
    y_pred_bin = (y_pred_test == c).astype(int)
    p = precision(y_true_bin, y_pred_bin)
    r = recall(y_true_bin, y_pred_bin)
    f = f1_score(y_true_bin, y_pred_bin)
    print(f"  {c:<6} {p:>10.3f} {r:>9.3f} {f:>8.3f}")

# Macro-average F1
macro_f1 = np.mean([
    f1_score((y_test == c).astype(int), (y_pred_test == c).astype(int))
    for c in range(N_CLASSES)
])
print(f"\nMacro-average F1 : {macro_f1:.4f}")

In [ ]:
# --- Confusion matrix ---
cm = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for t, p in zip(y_test, y_pred_test):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels([f"P:{c}" for c in range(N_CLASSES)], rotation=45, ha="right", fontsize=9)
ax.set_yticklabels([f"T:{c}" for c in range(N_CLASSES)], fontsize=9)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Confusion Matrix — Test Set", fontweight="bold")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if cm[i, j] > 0:
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    fontsize=7, color="white" if cm[i, j] > cm.max() * 0.5 else "black")
plt.tight_layout()
plt.show()

**Interpretation:** Strong diagonal entries indicate the model classifies most note classes correctly. Off-diagonal entries reveal confusion between specific classes. Adjacent classes (similar MIDI pitches) are typically the most confused — this is physically expected because their spectral fingerprints differ by roughly a semitone (~6% frequency shift). Classes that are an octave apart (12 semitones) share harmonic structure and may also be confused.

Compare train vs. test accuracy: a large gap signals **overfitting** — the model has memorised training data at the expense of generalisation. If both are low, the model is **underfitting** and needs more capacity or more training.

## 9. Sklearn Comparison

In [ ]:
sk_mlp = SklearnMLP(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="sgd",
    learning_rate_init=0.01,
    max_iter=80,
    random_state=SEED,
    batch_size=256,
    early_stopping=False,
)
sk_mlp.fit(X_train_sc, y_train)
sk_pred = sk_mlp.predict(X_test_sc)

sk_acc = accuracy(y_test, sk_pred)
sk_f1 = np.mean([
    f1_score((y_test == c).astype(int), (sk_pred == c).astype(int))
    for c in range(N_CLASSES)
])

print(f"{'Metric':<20} {'From Scratch':>14} {'Sklearn':>10}")
print("-" * 47)
print(f"{'Test Accuracy':<20} {test_acc:>14.4f} {sk_acc:>10.4f}")
print(f"{'Macro F1':<20} {macro_f1:>14.4f} {sk_f1:>10.4f}")
print()
print("Note: sklearn uses Adam optimizer and cross-entropy loss by default;")
print("our implementation uses SGD + MSE, so some gap is expected.")

**Interpretation:** Sklearn's MLPClassifier uses cross-entropy loss and Adam by default (better optimised for classification). Our from-scratch implementation uses MSE + SGD, which is mathematically valid but typically converges more slowly and to lower accuracy. The comparison shows the qualitative trend is correct — both models learn to classify notes — even if sklearn's tuned defaults outperform. Switching to cross-entropy loss and Adam in our implementation would close most of the gap.

## 10. Architecture Exploration

We compare three hidden-layer configurations to see how depth and width affect test accuracy. All other settings are held constant.

In [ ]:
configs = [
    ([32],      "[32]"),
    ([64, 32],  "[64, 32]"),
    ([128, 64, 32], "[128, 64, 32]"),
]

results = []
for hidden, label in configs:
    m = MLP(
        hidden_layers=hidden,
        activation=ReLU(),
        output_activation=ReLU(),
        loss=MeanSquaredError(),
        optimizer=SGD(learning_rate=0.01),
        n_epochs=80,
        batch_size=256,
        random_state=SEED,
    ).fit(X_train_sc, y_train_oh)
    preds = np.argmax(m.predict(X_test_sc), axis=1)
    acc_val = accuracy(y_test, preds)
    results.append((label, acc_val, m.loss_history_))
    print(f"{label:<20}  test acc = {acc_val:.4f}")

# Plot loss curves side by side
fig, ax = plt.subplots(figsize=(9, 4))
for label, _, hist in results:
    ax.plot(range(1, len(hist) + 1), hist, label=label, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training MSE")
ax.set_title("Loss Curves by Architecture", fontweight="bold")
ax.set_yscale("log")
ax.legend(title="Hidden layers")
plt.tight_layout()
plt.show()

**Interpretation:** Larger architectures generally achieve lower training loss and often better test accuracy, but with diminishing returns. A deeper network ([128, 64, 32]) has more parameters and can represent more complex functions — but also takes longer to converge and is more prone to overfitting on smaller datasets. For this 25K-frame dataset, the [64, 32] architecture is a reasonable balance. If training and test accuracy diverge as model size grows, regularisation (dropout, L2 weight decay) would be the next step.

## 11. Summary

| | |
|---|---|
| **Algorithm** | MLP with backpropagation |
| **Task** | 12-class MIDI note classification |
| **Features** | 18 audio features per 46 ms frame |
| **Training data** | 10 GuitarSet tracks, all 6 strings, voiced frames only |
| **Architecture** | [64, 32] hidden + 12-unit output |
| **Loss** | Mean Squared Error (one-hot targets) |
| **Optimiser** | SGD (η = 0.01, batch = 256) |

**Takeaways:**

1. The 12-class note classification task is substantially harder than binary voiced/unvoiced detection — the classes overlap in PCA space and no single hyperplane separates them, motivating the MLP's nonlinear hidden layers.
2. The loss curve should decrease smoothly; if it plateaus or oscillates, tuning the learning rate is the first lever.
3. Per-class F1 reveals which notes are hardest — typically those adjacent in pitch or sharing harmonics with other notes.
4. Sklearn's Adam + cross-entropy defaults outperform our SGD + MSE setup, illustrating why loss function choice and optimizer matter as much as architecture.
5. A natural extension is **polyphonic** classification — multiple notes simultaneously — which would require multi-label learning rather than multi-class.